In [1]:
import pandas as pd
import lightgbm as lgb
import shap
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

/home/rigonzal/miniconda3/envs/genv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import matplotlib
matplotlib.__version__

'3.8.4'

In [3]:
shap.__version__

'0.46.0'

# 0.Config

In [4]:
path_fe = "../../outputs/fe/"

# 1. Data

In [27]:
df_mdt = pd.read_parquet(f"{path_fe}train_all_joined_fe_delta_years=4.parquet")
df_mdt.shape

(1877383, 71)

# 2. Processing

In [28]:
def arrival_category(row):
    if row <=15:
        return "On-Time"
    elif row <=45:
        return "Late"
    else:
        return "Very Late"

In [29]:
# Create binary target
df_mdt["arrival_category"] = df_mdt["arrdelay"].apply(arrival_category)
df_mdt = df_mdt.drop(["arrdelay"], axis=1)
df_mdt["arrival_category"].value_counts(normalize=True)

arrival_category
On-Time      0.785923
Very Late    0.113150
Late         0.100927
Name: proportion, dtype: float64

In [30]:
df_mdt["dt_obj"] = df_mdt["month"].astype(str).str.slice(0,8)  + df_mdt["day"].astype(str)
df_mdt["dt_obj"] = pd.to_datetime(df_mdt["dt_obj"])
df_mdt["day_of_week"] = df_mdt["dt_obj"].dt.dayofweek
df_mdt["month"] = df_mdt["dt_obj"].dt.month

In [31]:
# binary On-Time vs Late

y_1 = (df_mdt["arrival_category"] != "On-Time").astype(int)
df_mdt_1 = df_mdt.drop(columns=["unique_key","arrival_category", "dt_obj"])

In [32]:
# binary Late vs Very Late
cond_ = df_mdt["arrival_category"].isin(["Late", "Very Late"])
y_2 = (df_mdt[cond_]["arrival_category"] == "Very Late").astype(int)
df_mdt_2 = df_mdt[cond_].drop(columns=["unique_key","arrival_category", "dt_obj"]).copy()

In [33]:
X_train1, X_val1, y_train1, y_val1 = train_test_split(df_mdt_1, y_1, stratify=y_1, test_size=0.1, random_state=42)
X_train2, X_val2, y_train2, y_val2 = train_test_split(df_mdt_2, y_2, stratify=y_2, test_size=0.1, random_state=42)

In [34]:
y_val1.mean(),y_val2.mean()

(np.float64(0.21407912048109343), np.float64(0.5285511681719788))

# 3. Train

In [35]:
from sklearn.metrics import classification_report

In [37]:
model1 = lgb.LGBMClassifier(
                          # is_unbalance='True',
                               verbose=-1,
                               num_leaves=20,
                               n_estimators=100,
                               max_depth=5,
                               colsample_bytree=0.8,
                               lambda_l1=1,
                               scale_pos_weight=4)

model1.fit(X_train1, y_train1)

LGBMClassifier(colsample_bytree=0.8, lambda_l1=1, max_depth=5, num_leaves=20,
               scale_pos_weight=4, verbose=-1)

In [38]:
model2 = lgb.LGBMClassifier(
                           is_unbalance='False',
                               verbose=-1,
                               num_leaves=20,
                               n_estimators=100,
                               max_depth=5,
                               colsample_bytree=0.8,
                               lambda_l1=1)

model2.fit(X_train2, y_train2)

LGBMClassifier(colsample_bytree=0.8, is_unbalance='False', lambda_l1=1,
               max_depth=5, num_leaves=20, verbose=-1)

# 4. Metrics

In [39]:
# 🔹 Predictions & Metrics
y_train_pred = model1.predict(X_train1)
y_val_pred = model1.predict(X_val1)

print(classification_report(y_train1, y_train_pred))

print(classification_report(y_val1, y_val_pred))

              precision    recall  f1-score   support

           0       0.89      0.64      0.74   1327931
           1       0.35      0.72      0.47    361713

    accuracy                           0.65   1689644
   macro avg       0.62      0.68      0.61   1689644
weighted avg       0.78      0.65      0.69   1689644

              precision    recall  f1-score   support

           0       0.89      0.64      0.74    147548
           1       0.35      0.72      0.47     40191

    accuracy                           0.66    187739
   macro avg       0.62      0.68      0.61    187739
weighted avg       0.78      0.66      0.69    187739



In [40]:
# 🔹 Predictions & Metrics
y_train_pred = model2.predict(X_train2)
y_val_pred = model2.predict(X_val2)

print(classification_report(y_train2, y_train_pred))

print(classification_report(y_val2, y_val_pred))

              precision    recall  f1-score   support

           0       0.60      0.51      0.55    170530
           1       0.61      0.69      0.65    191183

    accuracy                           0.61    361713
   macro avg       0.61      0.60      0.60    361713
weighted avg       0.61      0.61      0.60    361713

              precision    recall  f1-score   support

           0       0.59      0.50      0.54     18948
           1       0.60      0.69      0.64     21243

    accuracy                           0.60     40191
   macro avg       0.60      0.59      0.59     40191
weighted avg       0.60      0.60      0.59     40191



# 5. Final metrics

In [41]:
df_mdt = pd.read_parquet(f"{path_fe}val_all_joined_fe_delta_years=4.parquet")
df_mdt.shape

(469344, 71)

In [42]:
def arrival_category(row):

    if row <=15:
        return "On-Time"
    elif row <=45:
        return "Late"
    else:
        return "Very Late"

In [43]:
# Create binary target
df_mdt["arrival_category"] = df_mdt["arrdelay"].apply(arrival_category)
df_mdt = df_mdt.drop(["arrdelay"], axis=1)
df_mdt["arrival_category"].value_counts(normalize=True)

arrival_category
On-Time      0.785925
Very Late    0.113149
Late         0.100926
Name: proportion, dtype: float64

In [44]:
df_mdt["dt_obj"] = df_mdt["month"].astype(str).str.slice(0,8)  + df_mdt["day"].astype(str)
df_mdt["dt_obj"] = pd.to_datetime(df_mdt["dt_obj"])
df_mdt["day_of_week"] = df_mdt["dt_obj"].dt.dayofweek
df_mdt["month"] = df_mdt["dt_obj"].dt.month

In [45]:
# 🔹 Define features and target
y = df_mdt["arrival_category"].values.copy()
df_mdt = df_mdt.drop(columns=["unique_key","arrival_category", "dt_obj"])

In [47]:
pd.Series(y ).value_counts()

On-Time      368869
Very Late     53106
Late          47369
Name: count, dtype: int64

In [ ]:
y1_pred_val = model1.predict(df_mdt)
y2_pred_val = model2.predict(df_mdt)

y_pred = []
for i in range(len(y)):
    if y1_pred_val[i] == 1:
        if y2_pred_val[i] == 1:
            y_pred.append("Very Late")
        else:
            y_pred.append("Late")
    else:
        y_pred.append("On-Time")

In [51]:
print(classification_report(y, y_pred))

              precision    recall  f1-score   support

        Late       0.15      0.21      0.17     47369
     On-Time       0.89      0.64      0.74    368869
   Very Late       0.24      0.65      0.35     53106

    accuracy                           0.59    469344
   macro avg       0.43      0.50      0.42    469344
weighted avg       0.75      0.59      0.64    469344



In [64]:
y1_pred_val = model1.predict_proba(df_mdt)[:, 1]
y2_pred_val = model2.predict_proba(df_mdt)[:, 1]

y_pred = []
for i in range(len(y)):
    if y1_pred_val[i] > 0.7:
        if y2_pred_val[i] > 0.5:
            y_pred.append("Very Late")
        else:
            y_pred.append("Late")
    else:
        y_pred.append("On-Time")
print(classification_report(y, y_pred))

              precision    recall  f1-score   support

        Late       0.24      0.01      0.02     47369
     On-Time       0.83      0.93      0.88    368869
   Very Late       0.36      0.38      0.37     53106

    accuracy                           0.77    469344
   macro avg       0.48      0.44      0.42    469344
weighted avg       0.72      0.77      0.73    469344

